In [ ]:
## SENSITIVITY ANALYSIS for B_crit = 100

import numpy as np
from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde

# Helpers 
def pos(x):
    return (x + Abs(x)) / 2


def run_model_dde(params):

    # Fixed Parameters
    phi_B   = 0.018
    phi_H   = 0.007
    b       = 500
    a       = 500
    k       = 5000
    kappa   = 1/9
    tau     = 12
    gamma_B = 0.003
    theta   = 105
    
    # Varied Parameters
    (chi, sigma, epsilon_B, gamma_H, 
     delta_1, delta_2, epsilon_T, gamma_T ) = params

    # Initial Conditions
    B0, H0, T0, V0, f0 = 5000, 10000, 2, 1, 1000

    q0 = delta_1*T0/(a+B0) + delta_2*V0/(a+B0)
    I0 = q0 * tau

    # States
    B = y(0)
    H = y(1)
    T = y(2)
    V = y(3)
    f = y(4)
    I = y(5)

    B_tau = y(0, t - tau)
    T_tau = y(2, t - tau)
    V_tau = y(3, t - tau)

    # Nonnegativity
    Bp, Hp, Tp, Vp, fp, Ip = map(pos, (B, H, T, V, f, I))
    Btp, Ttp, Vtp = map(pos, (B_tau, T_tau, V_tau))

    # MODEL TERMS 
    S = (fp/(b+fp)) * (Hp/(k+Hp))

    infestation = delta_1 * (Bp/(a+Bp)) * Tp
    infection   = delta_2 * (Bp/(a+Bp)) * Vp

    q_now = delta_1*Tp/(a+Bp) + delta_2*Vp/(a+Bp)
    q_tau = delta_1*Ttp/(a+Btp) + delta_2*Vtp/(a+Btp)

    dI = q_now - q_tau
    exp_B = exp(-gamma_B*tau - Ip)

    Phi = chi + sigma*cos(2*pi*(t-theta)/365)

    dB = epsilon_B * S - infestation - infection - gamma_B * Bp - kappa * exp_B * Btp
    dH = kappa * exp_B * Btp - gamma_H * Hp
    dT = epsilon_T * infestation - gamma_T * Tp
    dV = epsilon_T * infection   - gamma_T * Vp
    df = Phi * (Hp/(k+Hp)) - phi_B * Bp - phi_H * Hp

    # Stabilization (important for PRCC robustness)
    lam = 120
    dB += lam*(Bp-B)
    dH += lam*(Hp-H)
    dT += lam*(Tp-T)
    dV += lam*(Vp-V)
    df += lam*(fp-f)
    dI += lam*(Ip-I)

    # BUILD SOLVER 
    DDE = jitcdde([dB, dH, dT, dV, df, dI])

    def history(s):
        return [B0, H0, T0, V0, f0, I0]

    DDE.past_from_function(history)
    DDE.step_on_discontinuities()
    DDE.set_integration_parameters(max_step=0.02)

    # Integration + brood-threshold outcome
    STARTTIME, STOPTIME, DT = 0, 365, 0.02
    B_crit = 100

    times = np.arange(STARTTIME + 1e-6, STOPTIME + DT/2, DT)

    Tcrit = STOPTIME  # default: "no brood depletion within horizon"
    for tt in times:
        B_val = DDE.integrate(tt)[0]   # B is index 0
        if B_val <= B_crit:
            Tcrit = float(tt)
            break

    return Tcrit

In [ ]:
## SENSITIVITY ANALYSIS for B_crit = 100
#Latin Hypercube Sampling + PRCC

from scipy.stats import qmc, spearmanr

N = 200

param_names = [ r"$\chi$", r"$\sigma$", r"$\epsilon_B$", r"$\gamma_H$",
    r"$\delta_1$", r"$\delta_2$", r"$\epsilon_T$", r"$\gamma_T$" ]

bounds = np.array([
    [500, 700],      # chi
    [200, 400],      # sigma
    [1500, 2000],    # epsilon_B
    [0.0, 0.17],     # gamma_H
    [0.0, 0.1],      # delta_1
    [0.0, 0.1],      # delta_2
    [1, 2.1],        #[0.79, 2.1],     # epsilon_T
    [1/28, 1/12],    # gamma_T    
    ])

sampler = qmc.LatinHypercube(d=bounds.shape[0]) #, seed=42)
#np.random.seed(42)
X = qmc.scale(sampler.random(N), bounds[:,0], bounds[:,1])

Y = np.array([run_model_dde(X[i]) for i in range(N)])


def PRCC(X, Y):
    p = X.shape[1]
    prcc = np.zeros(p)
    for i in range(p):
        Xi = X[:, i]
        Xr = np.delete(X, i, axis=1)

        r_xy, _ = spearmanr(Xi, Y)
        r_xr = np.array([spearmanr(Xi, Xr[:,j])[0] for j in range(p-1)])
        r_yr = np.array([spearmanr(Y,  Xr[:,j])[0] for j in range(p-1)])

        prcc[i] = (r_xy - np.dot(r_xr, r_yr)) / \
                  np.sqrt((1-np.dot(r_xr,r_xr))*(1-np.dot(r_yr,r_yr)))
    return prcc

prcc_values = PRCC(X, Y)

In [ ]:
## Figure 11. SENSITIVITY ANALYSIS for B_crit = 100
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 5))
H_color   = (255/255, 194/255, 10/255)
bars = plt.bar(param_names, prcc_values, color=H_color, edgecolor='black', linewidth=1.5)

plt.axhline(0, color='black', linewidth=1)
plt.title( "PRCC for time to brood threshold", fontsize=15 )
ax.set_ylabel("Sensitivity index", color='black', fontsize=15, labelpad=10, ha='center')
ax.set_xlabel("Parameter", color='black', fontsize=15, labelpad=10, ha='center')
ax.tick_params(axis='x', which='major', labelsize=18, width=1.6, length=6, colors='black')
ax.tick_params(axis='y', which='major', labelsize=18, width=1.6, length=6, colors='black')

# y-axis ticks and limits
ax.set_yticks([-1, -0.5, 0, 0.5, 1])
ax.set_ylim(-1, 1)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.text(-0.11, 1.15, "(a)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')
# value labels
for bar in bars:
    y = bar.get_height()
    plt.text( bar.get_x() + bar.get_width()/2, y + 0.03*np.sign(y),
        f"{y:.2f}", ha='center', va='bottom' if y >= 0 else 'top', fontsize=15 )
    
#plt.savefig("Sensitivity_B.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
##SENSITIVITY ANALYSIS for H_crit = 1000

import numpy as np
from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde

# Helpers 
def pos(x):
    return (x + Abs(x)) / 2


def run_model_dde(params):

    # Fixed Parameters
    phi_B   = 0.018
    phi_H   = 0.007
    b       = 500
    a       = 500
    k       = 5000
    kappa   = 1/9
    tau     = 12
    gamma_B = 0.003
    theta   = 105
    
    # Varied Parameters
    (chi, sigma, epsilon_B, gamma_H, 
     delta_1, delta_2, epsilon_T, gamma_T ) = params

    # Initial Conditions
    B0, H0, T0, V0, f0 = 5000, 10000, 2, 1, 1000

    q0 = delta_1*T0/(a+B0) + delta_2*V0/(a+B0)
    I0 = q0 * tau

    # States
    B = y(0)
    H = y(1)
    T = y(2)
    V = y(3)
    f = y(4)
    I = y(5)

    B_tau = y(0, t - tau)
    T_tau = y(2, t - tau)
    V_tau = y(3, t - tau)

    # nonnegativity
    Bp, Hp, Tp, Vp, fp, Ip = map(pos, (B, H, T, V, f, I))
    Btp, Ttp, Vtp = map(pos, (B_tau, T_tau, V_tau))

    # MODEL TERMS 
    S = (fp/(b+fp)) * (Hp/(k+Hp))

    infestation = delta_1 * (Bp/(a+Bp)) * Tp
    infection   = delta_2 * (Bp/(a+Bp)) * Vp

    q_now = delta_1*Tp/(a+Bp) + delta_2*Vp/(a+Bp)
    q_tau = delta_1*Ttp/(a+Btp) + delta_2*Vtp/(a+Btp)

    dI = q_now - q_tau
    exp_B = exp(-gamma_B*tau - Ip)

    Phi = chi + sigma*cos(2*pi*(t-theta)/365)

    dB = epsilon_B * S - infestation - infection - gamma_B * Bp - kappa * exp_B * Btp
    dH = kappa * exp_B * Btp - gamma_H * Hp
    dT = epsilon_T * infestation - gamma_T * Tp
    dV = epsilon_T * infection   - gamma_T * Vp
    df = Phi * (Hp/(k+Hp)) - phi_B * Bp - phi_H * Hp

    # Stabilization (important for PRCC robustness)
    lam = 120
    dB += lam*(Bp-B)
    dH += lam*(Hp-H)
    dT += lam*(Tp-T)
    dV += lam*(Vp-V)
    df += lam*(fp-f)
    dI += lam*(Ip-I)

    # BUILD SOLVER 
    DDE = jitcdde([dB, dH, dT, dV, df, dI])

    def history(s):
        return [B0, H0, T0, V0, f0, I0]

    DDE.past_from_function(history)
    DDE.step_on_discontinuities()
    DDE.set_integration_parameters(max_step=0.02)

    # Integration + collapse-time outcome
    STARTTIME, STOPTIME, DT = 0, 365, 0.02
    H_crit = 1000

    times = np.arange(STARTTIME + 1e-6, STOPTIME + DT/2, DT)

    Tcrit = STOPTIME  # default: "no collapse within horizon"
    for tt in times:
        H_val = DDE.integrate(tt)[1]
        if H_val <= H_crit:
            Tcrit = float(tt)
            break

    return Tcrit

In [ ]:
## SENSITIVITY ANALYSIS for H_crit = 100
#Latin Hypercube Sampling + PRCC

from scipy.stats import qmc, spearmanr

N = 200

param_names = [ r"$\chi$", r"$\sigma$", r"$\epsilon_B$", r"$\gamma_H$",
    r"$\delta_1$", r"$\delta_2$", r"$\epsilon_T$", r"$\gamma_T$" ]

bounds = np.array([
    [500, 700],      # chi
    [200, 400],      # sigma
    [1500, 2000],    # epsilon_B
    [0.0, 0.17],     # gamma_H
    [0.0, 0.1],      # delta_1
    [0.0, 0.1],      # delta_2
    [1, 2.1],        #[0.79, 2.1],     # epsilon_T
    [1/28, 1/12],    # gamma_T    
    ])

sampler = qmc.LatinHypercube(d=bounds.shape[0])#, seed=42)
#np.random.seed(42)
X = qmc.scale(sampler.random(N), bounds[:,0], bounds[:,1])

Y = np.array([run_model_dde(X[i]) for i in range(N)])


def PRCC(X, Y):
    p = X.shape[1]
    prcc = np.zeros(p)
    for i in range(p):
        Xi = X[:, i]
        Xr = np.delete(X, i, axis=1)

        r_xy, _ = spearmanr(Xi, Y)
        r_xr = np.array([spearmanr(Xi, Xr[:,j])[0] for j in range(p-1)])
        r_yr = np.array([spearmanr(Y,  Xr[:,j])[0] for j in range(p-1)])

        prcc[i] = (r_xy - np.dot(r_xr, r_yr)) / \
                  np.sqrt((1-np.dot(r_xr,r_xr))*(1-np.dot(r_yr,r_yr)))
    return prcc

prcc_values = PRCC(X, Y)

In [ ]:
##SENSITIVITY ANALYSIS for H_crit = 1000
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
H_color   = (255/255, 194/255, 10/255)
bars = plt.bar(param_names, prcc_values, color=H_color, edgecolor='black', linewidth=1.5)

plt.axhline(0, color='black', linewidth=1)
plt.title( "PRCC for time to hive bee threshold", fontsize=15 )
ax.set_ylabel("Sensitivity index", color='black', fontsize=15, labelpad=10, ha='center')
ax.set_xlabel("Parameter", color='black', fontsize=15, labelpad=10, ha='center')
ax.tick_params(axis='x', which='major', labelsize=18, width=1.6, length=6, colors='black')
ax.tick_params(axis='y', which='major', labelsize=18, width=1.6, length=6, colors='black')

# y-axis ticks and limits
ax.set_yticks([-1, -0.5, 0, 0.5, 1])
ax.set_ylim(-1, 1)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.text(-0.11, 1.15, "(b)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')
# value labels
for bar in bars:
    y = bar.get_height()
    plt.text( bar.get_x() + bar.get_width()/2, y + 0.03*np.sign(y),
        f"{y:.2f}", ha='center', va='bottom' if y >= 0 else 'top', fontsize=15 )
    
#plt.savefig("Sensitivity_H.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Combine Images
from PIL import Image, ImageOps
import numpy as np

def autocrop(im, pad=6):
    im = im.convert("RGB")
    arr = np.asarray(im).astype(np.int16)
    mask = np.any(arr < 245, axis=2)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return im
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(im.size[0], x1 + pad); y1 = min(im.size[1], y1 + pad)
    return im.crop((x0, y0, x1, y1))

def pad_to_height(im, target_h):
    w, h = im.size
    if h == target_h:
        return im
    top = (target_h - h)//2
    bottom = target_h - h - top
    return ImageOps.expand(im, border=(0, top, 0, bottom), fill="white")

# Load and crop 
imB = autocrop(Image.open("Sensitivity_B.png"), pad=6)
imH = autocrop(Image.open("Sensitivity_H.png"), pad=6)

# Match heights
target_h = max(imB.size[1], imH.size[1])
imB = pad_to_height(imB, target_h)
imH = pad_to_height(imH, target_h)

# Pure white space
gap = 40
space = Image.new("RGB", (gap, target_h), (255, 255, 255))  # WHITE

# Stitch
combined = Image.new("RGB", (imB.size[0] + gap + imH.size[0], target_h), (255, 255, 255))
combined.paste(imB, (0, 0))
combined.paste(space, (imB.size[0], 0))
combined.paste(imH, (imB.size[0] + gap, 0))

combined.save("Sensitivity.png", dpi=(600, 600))
print("Saved: Sensitivity.png")
